# SMS Spam Collection — pré-processamento

## Sobre o dataset

**SMS Spam Collection v.1** é um dataset de mensagens sobre SMS em inglês, cada uma rotulada como **`ham`** (mensagem legítima) ou **`spam`** (mensagem indesejada). Foi montado para pesquisa em filtragem de spam por **Tiago A. Almeida** (Unicamp) e **José María Gómez Hidalgo** (Universidad Europea de Madrid). Está disponível no [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).

> Almeida, T.A., Gómez Hidalgo, J.M., Yamakami, A. *Contributions to the Study of SMS Spam Filtering: New Collection and Results.* ACM DocEng 2011.

### De onde saíram as mensagens

| Fonte | Rótulo | Mensagens |
|---|---|---:|
| [Grumbletext](http://www.grumbletext.co.uk/): fórum britânico onde usuários denunciavam spam recebido (extração manual) | spam | 425 |
| [NUS SMS Corpus](https://www.comp.nus.edu.sg/~kanmy/): mensagens de voluntários, em sua maioria estudantes da National University of Singapore | ham | 3.375 |
| Tese de doutorado de Caroline Tagg (University of Birmingham) | ham | 450 |
| SMS Spam Corpus v.0.1 Big (Gómez Hidalgo) | ham / spam | 1.002 / 322 |

### Tamanho e distribuição

- **5.574 mensagens**: **4.827 ham (86,6%)** e **747 spam (13,4%)**. As classes são **desbalanceadas**, numa proporção de cerca de 6,5 ham para cada spam. Por isso a divisão treino/validação/teste é estratificada, e a acurácia sozinha engana: um modelo que sempre responde `ham` já acerta 86,6%.
- **Comprimento**: os spams são mais longos e mais uniformes.

| | caracteres (mediana) | caracteres (média ± dp) | palavras (mediana) | máx. caracteres |
|---|---:|---:|---:|---:|
| ham | 52 | 71,5 ± 58,3 | 11 | 910 |
| spam | 149 | 138,7 ± 28,9 | 25 | 223 |

  Os spams se concentram perto do limite de 160 caracteres de um SMS.
- **Duplicatas**: o corpus tem **403 mensagens repetidas** no texto bruto (439 depois do pré-processamento), na maioria spams de campanhas em massa e respostas curtas como "Ok". **Elas são mantidas de propósito**, porque refletem a distribuição real de mensagens. A contrapartida é que uma mesma mensagem pode cair no treino e no teste ao mesmo tempo.

## 1. Carregamento e inspeção


In [1]:
from collections import Counter
import csv
from sklearn.model_selection import train_test_split
import pandas as pd
import re
import html
import unicodedata
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from nltk.tokenize import TweetTokenizer
from urllib.parse import urlsplit
import tldextract


In [2]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    quoting=csv.QUOTE_NONE,
)

In [3]:
print("Total de mensagens:", len(df))
print("Mensagens repetidas (texto bruto):", df["message"].duplicated().sum())

comprimentos = df.assign(
    caracteres=df["message"].str.len(),
    palavras=df["message"].str.split().str.len(),
)
pd.concat([
    df["label"].value_counts().rename("mensagens"),
    df["label"].value_counts(normalize=True).mul(100).round(1).rename("%"),
    comprimentos.groupby("label")[["caracteres", "palavras"]].median().add_suffix(" (mediana)"),
], axis=1)

Total de mensagens: 5574
Mensagens repetidas (texto bruto): 403


,mensagens,%,caracteres (mediana),palavras (mediana)
label,,,,
ham,4827,86.6,52.0,11.0
spam,747,13.4,149.0,25.0


## 2. Inspeção de stopwords no dataframe


In [4]:
stop_words = set(ENGLISH_STOP_WORDS)
tokenizador_inspecao = TweetTokenizer(preserve_case=False, reduce_len=False)
stopwords_encontradas = Counter(
    token
    for message in df["message"].astype(str)
    for token in tokenizador_inspecao.tokenize(html.unescape(message))
    if token in stop_words
)
pd.DataFrame(stopwords_encontradas.most_common(), columns=["stopword", "ocorrencias"])


,stopword,ocorrencias
0,i,2351
1,to,2252
2,you,2149
3,a,1447
4,the,1338
...,...,...
266,whoever,1
267,beside,1
268,upon,1
269,afterwards,1


## Inspeção de pontuação antes do pré-processamento

A contagem usa as mensagens originais. Pontuação que aparece dentro de URLs também será contada. Símbolos especiais, como `$` e `+`, aparecem em uma lista separada.

In [5]:
mensagens_originais = df["message"].dropna().astype(str)

pontuacao_encontrada = Counter(
    caractere
    for mensagem in mensagens_originais
    for caractere in mensagem
    if unicodedata.category(caractere).startswith("P")
)

simbolos_encontrados = Counter(
    caractere
    for mensagem in mensagens_originais
    for caractere in mensagem
    if unicodedata.category(caractere).startswith("S")
)

print("Pontuacao encontrada (caractere: ocorrencias):")
for caractere, frequencia in pontuacao_encontrada.most_common():
    print(f"{caractere!r}: {frequencia}")

print("\nSimbolos especiais encontrados (caractere: ocorrencias):")
for caractere, frequencia in simbolos_encontrados.most_common():
    print(f"{caractere!r}: {frequencia}")


Pontuacao encontrada (caractere: ocorrencias):
'.': 11214
',': 1980
"'": 1879
'?': 1550
'!': 1397
'&': 922
';': 768
':': 745
'-': 585
')': 499
'/': 419
'"': 347
'*': 311
'#': 287
'(': 119
'@': 38
'‘': 37
'…': 16
'_': 15
'%': 10
'–': 9
'[': 5
']': 5
'\\': 4
'’': 3
'“': 2
'¡': 2
'»': 1
'—': 1

Simbolos especiais encontrados (caractere: ocorrencias):
'£': 329
'+': 137
'=': 47
'|': 36
'>': 31
'$': 23
'<': 6
'~': 3
'^': 1
'┾': 1


In [6]:
stop_words_para_preservar = {
    "no", "not", "nor", "cant", "cannot", "hasnt",
    "never", "nothing", "nobody", "none", "neither", "without",
    "call", "can", "now", "get", "please", "to", "your", "me",
}
stop_words_candidatas = {"a", "an", "the"}
stop_words_para_remover = stop_words_candidatas - stop_words_para_preservar

tokenizador_palavras = TweetTokenizer(preserve_case=True, reduce_len=False)


## 3. Função de pré-processamento

1. Decodificar HTML e converter para minúsculas.
2. Normalizar links e valores monetários nos formatos identificados.
3. Substituir contatos por marcadores, protegendo contrações, números, códigos e emoticons.
4. Limpar a pontuação restante e tokenizar com NLTK.
5. Restaurar os tokens protegidos e remover `a`, `an` e `the` por último.




In [7]:
extrator_dominio = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None)
sufixos_ambiguos = set(ENGLISH_STOP_WORDS) | {
    "call", "meet", "free", "love", "now", "win", "house", "pa",
    "ice", "you", "xxx", "onion", "hsbc", "life", "like", "im",
}
PADRAO_EMAIL = re.compile(
    r"(?<![\w.+-])[\w.+%-]+@[ \t]*(?:[a-z0-9-]+\.)+[a-z]{2,63}\b", re.I,
)
PADRAO_URL = re.compile(
    r"https?://[^\s<>\"'¡¿]+|www\.[^\s<>\"'¡¿]+|"
    r"(?<![\w@])(?:[a-z0-9-]+\.)+[a-z]{2,63}\b(?:[/?#][^\s<>\"'¡¿]*)?",
    re.I,
)
PADRAO_URL_ESPACADA = re.compile(
    r"((?:https?://|www\.)[^\s<>]*\.)[ \t]+([a-z0-9-]+)", re.I,
)
PADRAO_EMOTICON = r"(?:[:;=8][-^']?[dp](?![a-z])|[:;=8][-^']?[()]|:-[/\\]|<3(?!\d))"
PADRAO_TOKENS_PROTEGIDOS = re.compile(
    r"\[(?:URL|EMAIL|NUMBER|DECIMAL|TIME|UKP|PHONE|CAIXA-POSTAL)\]|[!?]{2,}|"
    + PADRAO_EMOTICON + r"|"
    r"\b[a-z]+(?:'[a-z]+)+\b|"
    r"(?:[£$€]|\bgbp)\d+(?:[.,]\d+)*(?:[a-z]+)?[%+]?|"
    r"(?<!\w)(?:[£$€]|gbp)?\+?\d+(?:[.,:/-]\d+)*(?:[a-z]+)?[%+]?(?!\w)|"
    r"\b[^\W\d_]\d+(?:[.,]\d+)+(?:[a-z]+)?\b|"
    r"\b(?=\w*\d)(?=\w*[a-z])\w+\b", re.I,
)

PADRAO_CAIXA_POSTAL = re.compile(
    r"(?<!\w)[a-z0-9]*p\.?[ \t]*o\.?[ \t]*box[ \t]*[,:#]?[ \t]*"
    r"(?=[a-z0-9/-]*\d)[a-z0-9]+(?:[/-][a-z0-9]+)*"
    r"(?:(?:[ \t]*[,.][ \t]*|[ \t]+)"
    r"(?:[a-z]{3,}(?:[ \t]*[,.][ \t]*|[ \t]+))?"
    r"(?:[a-z]{1,2}\d{1,2}[a-z]?[ \t]*[0-9o][a-z0-9]{2}|"
    r"[a-z]{1,2}\d[a-z]{2}|[a-z]{1,3}/[a-z]{1,2}\d{1,2})"
    r"[a-z0-9]*(?![\w/-]))?",
    re.I,
)


def substitui_caixa_postal(message):
    return PADRAO_CAIXA_POSTAL.sub(" [CAIXA-POSTAL] ", message)


def dominio_reconhecido(texto):
    partes = extrator_dominio(texto)
    return bool(partes.domain and partes.suffix)


def substitui_email(match):
    endereco = re.sub(r"@[ \t]+", "@", match.group())
    return "[EMAIL]" if dominio_reconhecido(endereco.rsplit("@", 1)[1]) else match.group()


def remove_url(match):
    link = match.group()
    final = ""
    while link and link[-1] in ".,!?;:)]}":
        final = link[-1] + final
        link = link[:-1]
    return "[URL]" + final


def substitui_url(match):
    candidato = match.group()
    if candidato.startswith(("http://", "https://", "www.")):
        return remove_url(match)
    partes = extrator_dominio(candidato)
    if not partes.domain or not partes.suffix:
        return candidato
    contexto = match.string[max(0, match.start() - 40):match.start()]
    tem_indicacao = bool(re.search(r"\b(?:visit|website|link|click|online|see|check)\b", contexto))
    tem_caminho = bool(re.search(r"[/?#]", candidato))
    if partes.suffix in sufixos_ambiguos and not (tem_indicacao or tem_caminho):
        return candidato
    return remove_url(match)


def normaliza_links_espacados(message):
    message = re.sub(r"\b(https?)//", r"\1://", message)
    message = re.sub(
        r"(\b(?:[a-z0-9-]+\.)+[a-z]{2,63}/[^\s<>]*\.)[ \t]+"
        r"(?=(?:html?|wml|php|asp)\b)", r"\1", message,
    )

    def juntar_partes(match):
        prefixo, continuacao = match.groups()
        completo = prefixo if prefixo.startswith("http") else "http://" + prefixo
        host = urlsplit(completo).hostname or ""
        if dominio_reconhecido(host.rstrip(".")):
            return match.group()
        return prefixo + continuacao

    while True:
        corrigida = PADRAO_URL_ESPACADA.sub(juntar_partes, message)
        if corrigida == message:
            return message
        message = corrigida


def normaliza_valores_monetarios(message):
    message = re.sub(r"([£$€]|\bgbp)[ \t]+(?=\d)", r"\1", message)
    message = re.sub(
        r"((?:[£$€]|\bgbp)\d+[.,])[ \t]+(\d{2})(?!\d)",
        r"\1\2", message,
    )
    message = re.sub(
        r"((?:[£$€]|\bgbp)\d+):[ \t]*(\d{2})(?!\d)",
        r"\1.\2", message,
    )
    return re.sub(
        r"\b(calls?)(?=\d+[.,]\d{2}(?:ppm|pm|p)\b)", r"\1 ", message,
    )


def protege_tokens(message):
    protegidos = {}

    def substituir(match):
        marcador = "zzprotegido" + "z" * (len(protegidos) + 1)
        while marcador in message or marcador in protegidos:
            marcador += "z"
        protegidos[marcador] = match.group()
        return " " + marcador + " "

    return PADRAO_TOKENS_PROTEGIDOS.sub(substituir, message), protegidos


In [8]:
def preprocess_message(message):
    message = html.unescape(str(message)).lower().replace("’", "'")
    message = normaliza_links_espacados(message)
    message = normaliza_valores_monetarios(message)

    message = PADRAO_EMAIL.sub(substitui_email, message)
    message = PADRAO_URL.sub(substitui_url, message)
    message = substitui_caixa_postal(message)
    for origem, destino in {
        "<#>": "[NUMBER]", "<decimal>": "[DECIMAL]", "<time>": "[TIME]",
        "<url>": "[URL]", "<email>": "[EMAIL]", "<ukp>": "[UKP]",
        "<fone no>": "[PHONE]",
    }.items():
        message = message.replace(origem, destino)
    message = re.sub(r"\[(url|email|number|decimal|time|ukp|phone|caixa-postal)\]",
                     lambda match: match.group().upper(), message)

    message, protegidos = protege_tokens(message)
    message = "".join(
        caractere
        if caractere.isalnum() or caractere.isspace() or caractere in "!?£$€%+@&"
        else " "
        for caractere in message
    )

    tokens = tokenizador_palavras.tokenize(message)
    tokens = [protegidos.get(token, token) for token in tokens]

    return [token for token in tokens if token not in stop_words_para_remover]


## 5. Aplicação e inspeção



In [9]:
df["message_preprocessed"] = (
    df["message"].astype(str).apply(preprocess_message).apply(lambda tokens: " ".join(tokens))
)

df.to_csv(
    "SMSSpamCollection_preprocessed.csv",
    index=False,
    sep=",",
    encoding="utf-8",
)

## 6. Divisão em treino, validação e teste




In [10]:
train_df, restante_df = train_test_split(
    df, test_size=0.30, random_state=42,
    stratify=df["label"],
)
validation_df, test_df = train_test_split(
    restante_df, test_size=0.50, random_state=42,
    stratify=restante_df["label"],
)

pd.DataFrame({
    "treino": train_df["label"].value_counts(),
    "validacao": validation_df["label"].value_counts(),
    "teste": test_df["label"].value_counts(),
}).fillna(0).astype(int)


,treino,validacao,teste
label,,,
ham,3378,724,725
spam,523,112,112


## 7. Teste rápido de atenção com BertViz

O modelo usado é o [`mrm8488/bert-tiny-finetuned-sms-spam-detection`](https://huggingface.co/mrm8488/bert-tiny-finetuned-sms-spam-detection), um BERT-tiny (2 camadas, 2 heads, cerca de 17 MB) já ajustado para spam de SMS. Ele roda em segundos na CPU. Para olhar um modelo maior, troque `MODELO` por `bert-base-uncased` (12 camadas × 12 heads, sem ajuste para spam).

A entrada é a **mensagem original** (`message`). O BERT tem tokenizador próprio, e os marcadores do pré-processamento (`[URL]`, `[NUMBER]`...) seriam quebrados em pedaços.

- `head_view`: escolha a camada e clique nas cores das heads. Passar o mouse sobre um token mostra para onde ele olha.
- `model_view`: mostra todas as camadas × heads de uma vez.

In [11]:
# Dependências: pip install torch transformers bertviz ipywidgets
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from bertviz import head_view, model_view

MODELO = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
tokenizador_bert = AutoTokenizer.from_pretrained(MODELO)
modelo_bert = AutoModelForSequenceClassification.from_pretrained(
    MODELO, output_attentions=True, attn_implementation="eager",
).eval()


def calcula_atencao(texto, max_length=64):
    entrada = tokenizador_bert(texto, return_tensors="pt", truncation=True, max_length=max_length)
    with torch.no_grad():
        saida = modelo_bert(**entrada)
    probabilidade_spam = saida.logits.softmax(-1)[0, 1].item()
    tokens = tokenizador_bert.convert_ids_to_tokens(entrada["input_ids"][0])
    return saida.attentions, tokens, probabilidade_spam

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

In [12]:
exemplo_spam = test_df[(test_df["label"] == "spam") & (test_df["message"].str.len() < 120)].iloc[0]
atencoes, tokens, probabilidade_spam = calcula_atencao(exemplo_spam["message"])
print(exemplo_spam["message"])
print(f"P(spam) = {probabilidade_spam:.3f}")
head_view(atencoes, tokens)

Mobile Club: Choose any of the top quality items for your mobile. 7cfca1a
P(spam) = 0.793


<IPython.core.display.Javascript object>

In [13]:
exemplo_ham = test_df[(test_df["label"] == "ham") & (test_df["message"].str.len().between(40, 100))].iloc[0]
atencoes, tokens, probabilidade_spam = calcula_atencao(exemplo_ham["message"])
print(exemplo_ham["message"])
print(f"P(spam) = {probabilidade_spam:.3f}")
model_view(atencoes, tokens)

If we win its really no 1 side for long time.
P(spam) = 0.062


<IPython.core.display.Javascript object>

### 7.1 Comparação: original × pré-processado × nosso tokenizador

O mesmo modelo recebe a mensagem em três formatos:

| Variante | Texto de entrada | Tokenização |
|---|---|---|
| `original` | mensagem bruta | tokenizador do BERT (WordPiece) |
| `pré-processado` | saída de `preprocess_message`, sem os marcadores (`[URL]`, `[NUMBER]`...) | tokenizador do BERT |
| `nosso tokenizador` | tokens de `preprocess_message` | **sem WordPiece**: cada token nosso vira um id do vocabulário do BERT. O que não existe no vocabulário, inclusive os marcadores, vira `[UNK]` |

**Cuidados na leitura:**
- O modelo foi ajustado com **texto bruto**. A comparação mede quanto cada formato se afasta do que ele viu no treino, e **não** qual pré-processamento seria melhor para treinar. Para isso, seria preciso ajustar um modelo em cada variante.
- O modelo foi ajustado no `sms_spam` do Hugging Face, que é este mesmo corpus. Ele provavelmente já viu as mensagens de teste, então os valores absolutos ficam otimistas. O que vale é a **diferença entre as variantes**.

In [14]:
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

PADRAO_MARCADOR = re.compile(r"\[(?:URL|EMAIL|NUMBER|DECIMAL|TIME|UKP|PHONE|CAIXA-POSTAL)\]")
vocabulario_bert = tokenizador_bert.get_vocab()


def texto_sem_marcadores(mensagem):
    return " ".join(token for token in preprocess_message(mensagem) if not PADRAO_MARCADOR.fullmatch(token))


VARIANTES = {
    "original": lambda mensagem: tokenizador_bert.tokenize(mensagem),
    "pré-processado": lambda mensagem: tokenizador_bert.tokenize(texto_sem_marcadores(mensagem)),
    "nosso tokenizador": preprocess_message,
}


def codifica(mensagem, variante, max_length=64):
    tokens = VARIANTES[variante](mensagem)[: max_length - 2]
    ids = [vocabulario_bert.get(token, tokenizador_bert.unk_token_id) for token in tokens]
    tokens = [tokenizador_bert.cls_token] + tokens + [tokenizador_bert.sep_token]
    ids = [tokenizador_bert.cls_token_id] + ids + [tokenizador_bert.sep_token_id]
    return tokens, torch.tensor([ids])


def preve(ids):
    with torch.no_grad():
        saida = modelo_bert(input_ids=ids)
    return saida.logits.softmax(-1)[0, 1].item(), saida.attentions

In [15]:
eh_spam = test_df["label"].eq("spam")
resultados = []
for variante in VARIANTES:
    probabilidades, quantidade_tokens, desconhecidos = [], 0, 0
    for mensagem in test_df["message"]:
        tokens, ids = codifica(mensagem, variante)
        probabilidades.append(preve(ids)[0])
        quantidade_tokens += len(tokens) - 2
        desconhecidos += (ids == tokenizador_bert.unk_token_id).sum().item()

    previsto_spam = pd.Series(probabilidades, index=test_df.index).gt(0.5)
    precisao, recall, f1, _ = precision_recall_fscore_support(
        eh_spam, previsto_spam, average="binary", zero_division=0,
    )
    resultados.append({
        "variante": variante,
        "acurácia": accuracy_score(eh_spam, previsto_spam),
        "precisão spam": precisao,
        "recall spam": recall,
        "F1 spam": f1,
        "tokens por mensagem": quantidade_tokens / len(test_df),
        "% [UNK]": 100 * desconhecidos / quantidade_tokens,
    })

pd.DataFrame(resultados).set_index("variante").round(3)

,acurácia,precisão spam,recall spam,F1 spam,tokens por mensagem,% [UNK]
variante,,,,,,
original,0.986,0.955,0.938,0.946,22.756,0.000
pré-processado,0.978,0.935,0.902,0.918,18.992,0.000
nosso tokenizador,0.877,0.909,0.089,0.163,15.571,14.594


In [16]:
exemplo = test_df[
    test_df["label"].eq("spam")
    & test_df["message_preprocessed"].str.contains(r"\[(?:URL|NUMBER|PHONE|UKP)\]")
    & test_df["message"].str.len().lt(130)
].iloc[0]["message"]
print(exemplo)

for variante in VARIANTES:
    tokens, ids = codifica(exemplo, variante)
    probabilidade_spam, atencoes = preve(ids)
    display(Markdown(f"#### {variante} — P(spam) = {probabilidade_spam:.3f}"))
    print(" ".join(tokens))
    head_view(atencoes, tokens)

"For the most sparkling shopping breaks from 45 per person; call 0121 2025050 or visit www.shortbreaks.org.uk"


#### original — P(spam) = 0.869

[CLS] " for the most sparkling shopping breaks from 45 per person ; call 01 ##21 202 ##50 ##50 or visit www . short ##break ##s . org . uk " [SEP]


<IPython.core.display.Javascript object>

#### pré-processado — P(spam) = 0.891

[CLS] for most sparkling shopping breaks from 45 per person call 01 ##21 202 ##50 ##50 or visit [SEP]


<IPython.core.display.Javascript object>

#### nosso tokenizador — P(spam) = 0.087

[CLS] for most sparkling shopping breaks from 45 per person call 0121 2025050 or visit [URL] [SEP]


<IPython.core.display.Javascript object>

## 8. Comparação justa: um modelo treinado em cada formato

A seção 7 usa um modelo que já foi ajustado com texto bruto. Qualquer outro formato chega a ele como algo que nunca viu, então o texto original leva vantagem por construção. Para saber se o pré-processamento ajuda, cada formato precisa de um modelo **treinado nele**, com o mesmo treino, a mesma validação e o mesmo teste.

Dois modelos:

1. **TF-IDF + regressão logística**: um modelo de saco de palavras, o tipo de modelo para o qual o pré-processamento foi pensado.
2. **BERT-tiny sem ajuste para spam** ([`google/bert_uncased_L-2_H-128_A-2`](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2)): a mesma arquitetura da seção 7, mas com os pesos do pré-treino genérico. Ele é ajustado no nosso treino, uma vez por formato e com 3 sementes, porque com 112 spams no teste um ou dois acertos mudam o F1. A época é escolhida pela validação.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

FORMATOS = {"original": "message", "pré-processado": "message_sem_marcadores"}
train_df, validation_df, test_df = (
    parte.assign(message_sem_marcadores=parte["message"].apply(texto_sem_marcadores))
    for parte in (train_df, validation_df, test_df)
)

y_treino = train_df["label"].eq("spam").to_numpy()
y_validacao = validation_df["label"].eq("spam").to_numpy()
y_teste = test_df["label"].eq("spam").to_numpy()


def metricas(previsto_spam):
    precisao, recall, f1, _ = precision_recall_fscore_support(
        y_teste, previsto_spam, average="binary", zero_division=0,
    )
    return {"precisão spam": precisao, "recall spam": recall, "F1 spam": f1}


resultados_tfidf = []
for formato, coluna in FORMATOS.items():
    vetorizador = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True)
    classificador = LogisticRegression(C=10, class_weight="balanced", max_iter=2000)
    classificador.fit(vetorizador.fit_transform(train_df[coluna]), y_treino)
    previsto_spam = classificador.predict(vetorizador.transform(test_df[coluna]))
    resultados_tfidf.append({"formato": formato, **metricas(previsto_spam)})

pd.DataFrame(resultados_tfidf).set_index("formato").round(3)

,precisão spam,recall spam,F1 spam
formato,,,
original,0.913,0.938,0.925
pré-processado,0.947,0.964,0.956


### Leitura dos resultados

- **No TF-IDF, o pré-processamento ajuda.** O F1 de spam sobe de cerca de 0,93 para 0,96. Um modelo de saco de palavras trata `£150`, `£200` e `£1000` como palavras diferentes que aparecem poucas vezes cada. O pré-processamento junta essas variações e limpa o ruído, e é para esse tipo de modelo que ele existe.
- **No BERT, os dois formatos empatam.** Com os dois treinados do mesmo jeito, os dois ficam em F1 entre 0,97 e 0,98, e a diferença é menor que a variação entre sementes. Numa execução anterior deste mesmo experimento o original ficou à frente (0,967 contra 0,956), e nesta o pré-processado ficou (0,982 contra 0,975). Quem ganha muda de uma rodada para outra, então não há ganho real de nenhum dos lados.
